# MiniMax H3 部署与运维

按顺序运行：1 启用 SSH（可选）→ 2 一键部署 → 3 查看进度。

部署完成后：本地执行 `ssh -L 8188:127.0.0.1:8188 root@<实例IP> -p <端口>`，浏览器打开 http://127.0.0.1:8188

## 0. 实例重启后恢复（幂等，可放心重复运行）

重启会丢失 overlay 上的内容（sshd、模型），但 /workspace 持久盘保留 ComfyUI 代码与 SSH 密钥。
运行下面单元格：重装 sshd + 恢复 SSH → 模型断点续传 → 启动 ComfyUI。

In [ ]:
!bash ../deploy/enable_ssh.sh
!bash ../deploy/setup_instance.sh

## 1. 启用 SSH（可选，需要外部终端/隧道时运行）

In [ ]:
!bash ../deploy/enable_ssh.sh

## 2. 一键部署（安装加速节点 → 魔搭下载模型约 64GB → 启动 ComfyUI，约 15-25 分钟）

In [ ]:
!bash ../deploy/setup_instance.sh

## 3. 查看进度与状态（可重复运行）

In [ ]:
!echo '--- 模型文件:'; du -sh /root/ComfyUI/models 2>/dev/null
!ls -la /root/ComfyUI/models/*/ 2>/dev/null | grep safetensors | awk '{print $5, $9}'
!echo '--- ComfyUI:'; curl -s -o /dev/null -w 'HTTP %{http_code}\n' --max-time 5 http://127.0.0.1:8188/ || echo 未运行
!echo '--- 日志尾部:'; tail -2 /root/comfyui.log 2>/dev/null

## 4. 启动 / 重启 ComfyUI

In [ ]:
!pkill -f 'main.py --listen' 2>/dev/null; sleep 3
!cd /root/ComfyUI && nohup /opt/venv/bin/python main.py --listen 0.0.0.0 --port 8188 > /root/comfyui.log 2>&1 &
!sleep 25; curl -s -o /dev/null -w 'HTTP %{http_code}\n' --max-time 5 http://127.0.0.1:8188/

## 5. 系统信息（GPU / 磁盘 / torch）

In [ ]:
!rocm-smi --showproductname --showmeminfo vram 2>/dev/null | grep -E 'Card Series|GFX Version|VRAM Total Memory'
!df -h / | tail -1
!/opt/venv/bin/python -c "import torch; print('torch', torch.__version__, '| GPU 数量', torch.cuda.device_count())"

## 6. 可选：下载 ref2va 参考图生视频权重（约 21GB，先确认磁盘余量足够！）

In [ ]:
!REF2VA=1 bash ../deploy/download_models_modelscope.sh /root/ComfyUI

## 7. 更新加速包（拉取 GitHub 最新代码并重装节点）

In [ ]:
!cd .. && (git pull || git -c url.'https://gh-proxy.com/https://github.com/'.insteadOf='https://github.com/' pull)
!/opt/venv/bin/python ../tools/install_into_comfyui.py --comfyui /root/ComfyUI --force